# Extract A-J from Google ASL Fingerspelling Dataset

Extracts isolated A-J letter sequences from the Google ASL Fingerspelling competition dataset.

**Output:**  files of shape (30, 126) per letter, saved to Google Drive.

## Before you start
1. Go to https://www.kaggle.com/competitions/asl-fingerspelling
2. Click Join Competition and accept the rules
3. Create a Kaggle API token (Settings -> API -> Create New Token)
4. Upload kaggle.json when prompted below


In [ ]:
# Install required packages
!pip install -q kaggle pandas numpy pyarrow mediapipe tqdm


In [ ]:
from google.colab import files
print("Upload your kaggle.json file (from Kaggle Settings -> API):")
uploaded = files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
print("Kaggle configured!")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Downloading train.csv...")
!kaggle competitions download -c asl-fingerspelling -f train.csv
!unzip -q -o train.csv.zip

df = pd.read_csv("train.csv")
print(f"Total sequences: {len(df)}")
df.head()


In [ ]:
# Filter for sequences with only A-J letters
target_letters = set("ABCDEFGHIJ")

def is_valid(phrase):
    if pd.isna(phrase) or len(str(phrase).strip()) == 0:
        return False
    p = str(phrase).upper().replace(" ", "")
    return all(c in target_letters for c in p)

filtered = df[df["phrase"].apply(is_valid)].copy()
print(f"Valid sequences: {len(filtered)}")
for ch in "ABCDEFGHIJ":
    count = sum(ch in str(p).upper() for p in filtered["phrase"])
    print(f"  {ch}: appears in {count} sequences")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = Path("/content/drive/MyDrive/sign_o_text_a_j")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for ch in "ABCDEFGHIJ": (OUTPUT_DIR / ch).mkdir(parents=True, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")


In [ ]:
# WARNING: This downloads ~2GB from Kaggle. Ensure you have disk space.

# The download may take 5-15 minutes depending on your connection.

import kagglehub
print("Downloading dataset...")
path = kagglehub.competition_download("asl-fingerspelling")
print(f"Dataset path: {path}")


In [ ]:
import pyarrow.parquet as pq
from tqdm import tqdm

SEQUENCE_LENGTH = 30
LANDMARK_DIR = Path(path) / "train_landmarks"
print(f"Landmarks: {LANDMARK_DIR}")

# Hand landmarks only (no pose/face padding) - saves ~90% storage

def process_file(file_id, phrase):
    parquet_path = LANDMARK_DIR / f"{file_id}.parquet"
    if not parquet_path.exists(): return 0
    try: table = pq.read_table(parquet_path)
    except Exception: return 0
    df_lm = table.to_pandas()
    total_frames = len(df_lm)
    p = str(phrase).upper().replace(" ", "")
    n_chars = len(p)
    if total_frames < SEQUENCE_LENGTH or n_chars == 0: return 0
    fpl = total_frames // n_chars
    created = 0
    for ci, ch in enumerate(p):
        if ch not in "ABCDEFGHIJ": continue
        start = ci * fpl
        end = start + fpl
        if end - start < SEQUENCE_LENGTH: continue
        seg = df_lm.iloc[start:end].values.astype(np.float32)
        for ws in range(0, len(seg) - SEQUENCE_LENGTH + 1, SEQUENCE_LENGTH // 2):
            win = seg[ws:ws + SEQUENCE_LENGTH]
            kps = np.zeros((SEQUENCE_LENGTH, 126), dtype=np.float16)
            for fi in range(SEQUENCE_LENGTH):
                frm = win[fi]
                lh = frm[:63] if len(frm) >= 63 else np.zeros(63)
                rh = frm[63:126] if len(frm) >= 126 else np.zeros(63)
                kps[fi] = np.concatenate([lh, rh]).astype(np.float16)
            (OUTPUT_DIR / ch).mkdir(exist_ok=True)
            np.savez_compressed(OUTPUT_DIR / ch / f"{file_id}_{ci}_w{ws:03d}.npz", hand=kps)
            created += 1
    return created


In [ ]:
MAX_PER_LETTER = 500
file_ids = filtered["file_id"].unique()
print(f"Processing {len(file_ids)} files...")
import random; random.shuffle(file_ids)
totals = {c: 0 for c in "ABCDEFGHIJ"}
for fid in tqdm(file_ids):
    if all(v >= MAX_PER_LETTER for v in totals.values()): break
    phrase = filtered[filtered["file_id"] == fid]["phrase"].iloc[0]
    process_file(fid, phrase)
    for c in "ABCDEFGHIJ":
        d = OUTPUT_DIR / c
        if d.exists(): totals[c] = len(list(d.glob("*.npz")))
print(f"
Final counts:{totals}")


In [ ]:
print("Verifying...")
for ld in sorted(OUTPUT_DIR.iterdir()):
    if ld.is_dir():
        nf = list(ld.glob("*.npz"))
        if nf: print(f"  {ld.name}: {len(nf)} files, shape={list(np.load(nf[0]).values())[0].shape}")
print("Done! Data in Google Drive.")


## Next steps
1. Download A-J folders from Drive to your project data/landmarks/
2. Get WLASL data for: hello, thanks, Father, Mother, Yes, No, Help
3. Run the retraining notebook
